# 🦖 Dino Run — PPO training on Google Colab

This notebook trains a **PPO** agent to play the *existing* Pygame Dino Run game.
It does not re-implement anything: the environment is a headless wrapper around the very
same simulation that the human-playable `main.py` runs. The game, the physics, the
observation, the reward and the action space are **unchanged** — this notebook only
orchestrates training on a Colab machine.

Repository: <https://github.com/Creative-banda/Dino-Run>

**Runtime ▸ Change runtime type ▸ T4 GPU** (a CPU runtime also works, just slower — the
notebook detects what it actually got instead of assuming).

## Order of work

| | step | why |
|---|---|---|
| 0 | configuration | the single cell you edit |
| A | clone / update the repository | always runs the *current* code from GitHub |
| B | detect GPU / CPU / RAM | nothing is assumed about the machine |
| C | install dependencies | `pip install -r requirements.txt` |
| B2 | mount Google Drive | the long run writes straight into Drive |
| D | headless probe | is pygame really running without a display? |
| E | benchmark parallel environments | the environment count is *measured*, never guessed |
| F | smoke test | the gate: do not continue if this fails |
| G | short PPO run (200k–500k steps) | proves Colab can actually train |
| H | evaluate the short model | mean / median / best score, survival, baselines |
| I | inspect the Drive run | resume state, steps trained, next saves |
| J | **long training (manual)** | the expensive cell, at the end, resumable |
| K | evaluate the final model | same evaluation code, fixed seeds |
| L | package + download | bring the model home |
| M | run it in the real game locally | the exact commands |

## After a Colab disconnect

`/content` is gone but Google Drive is not. Reopen this notebook, then run
**0 → A → B → C → E** (the benchmark must be re-run: the new runtime may be a different
machine), then **B2 → I → J**. Section J finds `latest.zip` inside the Drive folder and
continues from where it stopped — `--timesteps` is the *additional* budget, and the cell
computes the remainder from `run_config.json`.


## 0. Configuration

Everything tunable lives in this cell. The PPO values are the configuration this project
was verified with locally; they are passed explicitly so the defaults in
`training/train_ppo.py` can never drift silently.

In [ ]:
# =====================================================================================
# 0. CONFIGURATION -- the cell to edit.
#    Nothing here changes the game, the observation, the reward or the action space.
# =====================================================================================
import json
import os
import pathlib
import re
import shutil
import subprocess
import sys
import time

REPO_URL    = "https://github.com/Creative-banda/Dino-Run.git"
REPO_BRANCH = "main"
REPO_DIR    = "/content/Dino-Run"                     # code (re-created by cell A)
WORK_DIR    = "/content/dino_run_work"                # scratch: probe, smoke test, benchmark
DRIVE_BASE        = "/content/drive/MyDrive/dino_run_ppo"   # persistent experiment history
DRIVE_OUTPUT_DIR  = f"{DRIVE_BASE}/ppo_dino_colab"          # the long training run lives here (rename freely)
DRIVE_LOGS_DIR    = f"{DRIVE_BASE}/logs"                    # per-session console logs + benchmark/smoke reports (small)
SEED        = 42

# --- environment (unchanged) ---------------------------------------------------------
FRAME_SKIP        = 4        # agent steps per game frame (the verified setting)
MAX_EPISODE_STEPS = 6000     # agent steps per episode (6000 x 4 frames = 400 s of game time)

# --- PPO (the verified configuration -- change only deliberately) --------------------
LEARNING_RATE = 5e-4
N_STEPS       = 512
BATCH_SIZE    = 512
N_EPOCHS      = 6
GAMMA         = 0.997
GAE_LAMBDA    = 0.95
CLIP_RANGE    = 0.2
ENT_COEF      = 0.005
VF_COEF       = 0.5
TARGET_KL     = 0.05
NET_ARCH      = "64,64"

# --- run sizes -----------------------------------------------------------------------
SHORT_TIMESTEPS = 300_000        # 200k-500k: enough to prove Colab works
EVAL_EPISODES   = 5              # episodes per evaluation *during* training
FINAL_EPISODES  = 20             # episodes for the final evaluation (as used locally)
USE_TENSORBOARD = False          # True -> SB3 event files land in {DRIVE_OUTPUT_DIR}/tensorboard
LONG_OUTPUT     = DRIVE_OUTPUT_DIR              # the long run trains directly into Google Drive
SHORT_OUTPUT    = f"{DRIVE_OUTPUT_DIR}_short"   # the wiring-check run, also kept on Drive
BENCH_JSON      = f"{WORK_DIR}/benchmark_envs.json"

PY = f'"{sys.executable}"'      # quoted interpreter path: survives spaces in the path

pathlib.Path(WORK_DIR).mkdir(parents=True, exist_ok=True)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)      # lets the notebook import the project's own modules


# =====================================================================================
# helpers used by every cell below
# =====================================================================================
def run(cmd, cwd=REPO_DIR, check=True):
    """Run a shell command from the repository root and stop on failure."""
    print("$", cmd, flush=True)
    started = time.time()
    proc = subprocess.run(cmd, shell=True, cwd=cwd)
    if check and proc.returncode != 0:
        raise RuntimeError(f"command failed with exit code {proc.returncode}: {cmd}")
    print(f"[exit {proc.returncode} in {time.time() - started:.1f}s]", flush=True)
    return proc


def run_logged(cmd, log_path, cwd=REPO_DIR):
    """Run a command whose output goes to a log file; print the tail, then the verdict."""
    log = pathlib.Path(log_path)
    log.parent.mkdir(parents=True, exist_ok=True)
    proc = run(f'{cmd} > "{log}" 2>&1', cwd=cwd, check=False)
    text = log.read_text(errors="replace") if log.is_file() else ""
    print("\n".join(text.splitlines()[-35:]))
    print(f"[full log: {log}]")
    if proc.returncode != 0:
        raise RuntimeError(f"command failed with exit code {proc.returncode} -- read {log}")
    return text


def ppo_flags(learning_rate=LEARNING_RATE):
    """All PPO / environment flags in one place, so every run uses the same config."""
    parts = [
        f"--frame-skip {FRAME_SKIP}",
        f"--max-episode-steps {MAX_EPISODE_STEPS}",
        f"--n-steps {N_STEPS}",
        f"--batch-size {BATCH_SIZE}",
        f"--n-epochs {N_EPOCHS}",
        f"--gamma {GAMMA}",
        f"--gae-lambda {GAE_LAMBDA}",
        f"--clip-range {CLIP_RANGE}",
        f"--ent-coef {ENT_COEF}",
        f"--vf-coef {VF_COEF}",
        f"--target-kl {TARGET_KL}",
        f"--net-arch {NET_ARCH}",
    ]
    if learning_rate is not None:      # None = keep whatever the checkpoint used
        parts.append(f"--learning-rate {learning_rate}")
    return " ".join(parts)


def checkpoint_steps(path):
    """Step count encoded in a checkpoint name (dino_ppo_3000000_steps.zip -> 3000000)."""
    match = re.search(r"(\d+)_steps", pathlib.Path(path).name)
    return int(match.group(1)) if match else None


def locate_resume(output_dir):
    """Where a restarted runtime should continue from, and how far the run already got."""
    out = pathlib.Path(output_dir)
    config = {}
    if (out / "run_config.json").is_file():
        config = json.loads((out / "run_config.json").read_text())
    latest = out / "latest.zip"
    best = out / "best" / "best_model.zip"
    periodic = sorted((out / "checkpoints").glob("*.zip")) if (out / "checkpoints").is_dir() else []

    resume_from, source = None, "fresh run (nothing saved yet)"
    if latest.is_file():
        resume_from, source = latest, "latest.zip (state at the end of the last session)"
    elif periodic:
        resume_from = max(periodic, key=lambda p: p.stat().st_mtime)
        source = "newest periodic checkpoint (no latest.zip)"
    elif best.is_file():
        resume_from, source = best, "best/best_model.zip (no resumable checkpoint)"

    trained = max(
        [config.get("trained_timesteps") or 0]
        + [checkpoint_steps(p) or 0 for p in periodic]
        + [0]
    )
    return {
        "dir": out,
        "resume_from": resume_from,
        "source": source,
        "trained": int(trained),
        "config": config,
        "periodic": len(periodic),
        "latest": latest,
        "best": best,
    }


def summarize_evaluation(json_path, title="evaluation"):
    """Print the numbers this project reports: score mean/median/best, survival, baselines."""
    path = pathlib.Path(json_path)
    data = json.loads(path.read_text())
    settings = data.get("settings", {})
    results = {r["policy"]: r for r in data.get("results", []) if not r.get("error")}
    agent = results.get("ppo")
    print("=" * 82)
    print(title)
    print("=" * 82)
    if agent:
        print(f"episodes         : {agent['episodes']} deterministic, seeds {settings.get('seed')}..")
        print(f"score mean       : {agent['score_mean']:,.1f}   (std {agent['score_std']:,.1f})")
        print(f"score median     : {agent['score_median']:,.1f}")
        print(f"score best/worst : {agent['score_best']:,.0f} / {agent['score_worst']:,.0f}")
        print(f"survival mean    : {agent['survival_seconds_mean']:.1f} s "
              f"(best {agent['survival_seconds_best']:.1f} s)")
        print(f"distance mean    : {agent['distance_mean']:,.0f}")
        print(f"collisions       : {agent['collisions']}/{agent['episodes']} "
              f"(the rest hit the {settings.get('max_episode_steps')}-step cap)")
        print(f"actions          : {agent['action_distribution']}")
    for name, result in results.items():
        if name == "ppo":
            continue
        print(f"baseline {name:<20}: mean {result['score_mean']:>9,.1f}  "
              f"median {result['score_median']:>9,.1f}  survival {result['survival_seconds_mean']:>6.1f} s")
    for name, comparison in (data.get("comparison") or {}).items():
        print(f"  vs {name:<20} score x{comparison['score_ratio']:>7.1f}   "
              f"survival x{comparison['survival_ratio']:>7.1f}")
    print(f"raw json         : {path}")
    print("=" * 82)
    return data


def persistent_status():
    """What a fresh runtime wants to know about the persistent run on Drive."""
    state = locate_resume(LONG_OUTPUT)
    return {
        "drive_mounted": pathlib.Path("/content/drive/MyDrive").is_dir(),
        "drive_output_dir": LONG_OUTPUT,
        "drive_logs_dir": DRIVE_LOGS_DIR,
        "checkpoint_found": state["resume_from"] is not None,
        "resume_source": state["source"],
        "steps_trained": int(state["trained"]),
        "periodic_checkpoints": state["periodic"],
        "latest_zip": bool(state["latest"].is_file()),
        "best_model_zip": bool(state["best"].is_file()),
    }


def write_session_manifest(record):
    """Append one JSON line per training session to the persistent history (tiny file)."""
    manifest = pathlib.Path(LONG_OUTPUT) / "session_manifest.jsonl"
    manifest.parent.mkdir(parents=True, exist_ok=True)
    with manifest.open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(record, default=str) + "\n")
    print("session manifest :", manifest)
    return manifest


print("configuration loaded | repo:", REPO_DIR, "| work:", WORK_DIR, "| long-run output (Drive):", LONG_OUTPUT)


## A. Get the code (clone / update)

Clones the public repository, or brings an existing clone up to date with `origin/main`.
This is a **hard reset**: it makes the runtime match GitHub exactly and discards anything
you edited inside the repository here. Keep your own work in `WORK_DIR` or on Drive.

In [ ]:
# A. Clone the public repository, or update the existing clone to origin/main.
if (pathlib.Path(REPO_DIR) / ".git").is_dir():
    run(f"git fetch --depth 1 origin {REPO_BRANCH}")
    run(f"git checkout -f {REPO_BRANCH}")
    run(f"git reset --hard origin/{REPO_BRANCH}")
    print("updated existing clone ->", REPO_DIR)
else:
    if pathlib.Path(REPO_DIR).exists():
        shutil.rmtree(REPO_DIR)
    run(f"git clone --depth 1 --branch {REPO_BRANCH} {REPO_URL} {REPO_DIR}", cwd="/content")
    print("cloned", REPO_URL, "->", REPO_DIR)

commit = subprocess.run("git rev-parse HEAD", shell=True, cwd=REPO_DIR,
                        capture_output=True, text=True).stdout.strip()
print("commit:", commit)
run('git log -1 --format="%h %ad %s" --date=short')

# The notebook depends on the RL files being in the clone -- fail loudly instead of
# training something older than expected.
required = [
    "main.py",
    "requirements.txt",
    "game/core.py",
    "game/headless.py",
    "game/config.py",
    "rl/environment.py",
    "rl/observations.py",
    "rl/rewards.py",
    "rl/vector_env.py",
    "rl/process_vec_env.py",
    "training/train_ppo.py",
    "training/benchmark_envs.py",
    "training/evaluate_ppo.py",
    "training/smoke_test.py",
    "training/play_agent.py",
    "training/artifacts.py",
]
missing = [name for name in required if not (pathlib.Path(REPO_DIR) / name).is_file()]
if missing:
    raise RuntimeError(f"the clone is missing {missing} -- is the RL code pushed to {REPO_BRANCH}?")
print(f"all {len(required)} required files present")

## B. Detect the runtime (GPU / CPU / RAM)

Nothing about the machine is assumed: the GPU name and VRAM, the number of vCPUs, the
available RAM and the free disk are read from the runtime. A CPU-only runtime works too —
the policy is a tiny MLP, so the CPU (running the pygame simulation) is what limits
throughput.

In [ ]:
# B. What did we actually get?  GPU, CPU cores, RAM, disk.
import platform

ROOT = "/content" if pathlib.Path("/content").exists() else "/"
print("python        :", sys.version.split()[0], "|", platform.platform())
print("vCPU (logical):", os.cpu_count())
try:
    import psutil
    memory = psutil.virtual_memory()
    print("vCPU (physical):", psutil.cpu_count(logical=False))
    print(f"RAM           : {memory.total / 2 ** 30:.1f} GiB total, "
          f"{memory.available / 2 ** 30:.1f} GiB available")
except ImportError:
    print("RAM           : psutil not installed yet -- run section C")
print(f"disk {ROOT:<8} : {shutil.disk_usage(ROOT).free / 2 ** 30:.1f} GiB free")

CPU_CORES = os.cpu_count() or 2
try:
    import torch
    HAS_CUDA = bool(torch.cuda.is_available())
    print("torch         :", torch.__version__, "| built for CUDA:", torch.version.cuda)
    if HAS_CUDA:
        props = torch.cuda.get_device_properties(0)
        print(f"GPU           : {props.name} -- {props.total_memory / 2 ** 30:.1f} GiB, "
              f"compute capability {props.major}.{props.minor}")
        print("note          : the GPU only runs the policy network; pygame physics stays on the CPU")
    else:
        HAS_CUDA = False
        print("GPU           : none visible -> PPO trains on the CPU (supported, just slower)")
except ImportError:
    HAS_CUDA = False
    print("torch         : not installed yet (the Colab image ships it)")

IN_COLAB = pathlib.Path("/content").exists() and "google.colab" in sys.modules or pathlib.Path("/content").exists()
print("colab runtime :", IN_COLAB, "| GPU present:", HAS_CUDA)

## C. Install dependencies

`requirements.txt` asks for `torch>=2.0` (already satisfied by Colab's CUDA build) plus
pygame, gymnasium, stable-baselines3, numpy, psutil and cloudpickle. The installed
versions are printed so the report is reproducible.

In [ ]:
# C. Dependencies.  Colab's torch is already a CUDA build and requirements.txt only asks
#    for torch>=2.0, so pip leaves it alone.  Never install a CPU-only torch here.
run(f"{PY} -m pip install -q --upgrade-strategy only-if-needed -r requirements.txt")

import importlib

for name in ("pygame", "gymnasium", "stable_baselines3", "numpy", "psutil", "cloudpickle", "torch"):
    module = importlib.import_module(name)
    print(f"{name:<18} {getattr(module, '__version__', '?')}")

import torch
print("torch cuda build:", torch.version.cuda, "| cuda available:", torch.cuda.is_available())
gpu_attached = os.environ.get("COLAB_GPU") not in (None, "", "0")
if gpu_attached and not torch.cuda.is_available():
    print("WARNING: a GPU is attached to this runtime but torch cannot use it -- most likely a")
    print("         CPU-only torch wheel was installed. Fix: Runtime > Disconnect and delete")
    print("         runtime, then re-run this notebook without installing torch yourself.")

## B2. Mount Google Drive (persistent experiment history)

The long training run and the short wiring-check run write **directly into Drive**
(`DRIVE_OUTPUT_DIR`), so checkpoints, logs and metrics survive a disconnect. `/content`
stays the fast scratch disk for the high-frequency simulation work — only checkpoints,
logs, reports and metrics are written to Drive, never per-step data.

Re-running this cell is harmless. It fails loudly here if Drive cannot be mounted or
written to, rather than mid-training.


In [ ]:
# B2. Mount Drive, create the persistent folders, prove they are writable.
DRIVE_MOUNT = "/content/drive"

if not (pathlib.Path(DRIVE_MOUNT) / "MyDrive").is_dir():
    try:
        from google.colab import drive
    except ImportError as exc:
        raise RuntimeError(
            "google.colab is unavailable -- this notebook is meant to run on Google Colab."
        ) from exc
    drive.mount(DRIVE_MOUNT)
    print("Drive mounted at", DRIVE_MOUNT)
else:
    print("Drive is already mounted at", DRIVE_MOUNT)

for folder in (DRIVE_BASE, DRIVE_OUTPUT_DIR, DRIVE_LOGS_DIR):
    pathlib.Path(folder).mkdir(parents=True, exist_ok=True)

# a real write: a read-only or quota-exhausted Drive must fail here, not mid-training
_probe = pathlib.Path(DRIVE_BASE) / ".dino_write_probe"
_probe.write_text("ok")
_probe.unlink()

try:
    free_gib = shutil.disk_usage(DRIVE_BASE).free / 2 ** 30
    print(f"drive free space : {free_gib:.1f} GiB")
except OSError as exc:
    free_gib = None
    print("drive free space : unknown (", exc, ")")

DRIVE_STATUS = persistent_status()
DRIVE_STATUS["free_gib"] = round(free_gib, 1) if free_gib is not None else None

print("persistent output:", DRIVE_OUTPUT_DIR)
print("persistent logs  :", DRIVE_LOGS_DIR)
if DRIVE_STATUS["checkpoint_found"]:
    print("existing run     :", f"{DRIVE_STATUS['steps_trained']:,} steps trained "
          f"({DRIVE_STATUS['resume_source']})")
else:
    print("existing run     : nothing yet -- the long run will start fresh")


## D. Verify the environment (headless probe)

Runs the *real* environment: SDL `dummy` video driver (no window), no audio, no keyboard.
It checks the observation shape and range, the action meanings, and that stepping the
simulation works.

In [ ]:
# D. Headless probe: same game code, no display, no audio, no keyboard.
PROBE = """
import sys

sys.path.insert(0, sys.argv[1])      # the repository root, passed in by the caller

import numpy as np
from game.headless import describe_display
from rl.environment import DinoRunEnv

env = DinoRunEnv(frame_skip=4, seed=3)
obs, info = env.reset(seed=3)
print("display    :", describe_display())
description = env.describe()
print("env / obs  :", description.get("env_version"), description.get("obs_version"),
      "| size", description.get("observation_size"), "| shape", obs.shape, obs.dtype,
      "| inside space:", env.observation_space.contains(obs))
print("geometry   :", description.get("game_config", {}).get("width"), "x",
      description.get("game_config", {}).get("height"), "| frame_skip", description.get("frame_skip"))
print("actions    :", description.get("action_space", {}).get("meanings"), "| Discrete",
      description.get("action_space", {}).get("n"))
print("reward     :", description.get("reward_formula"))
total, resets = 0.0, 0
for _ in range(300):
    obs, reward, terminated, truncated, info = env.step(1)   # 1 = jump
    total += reward
    if terminated or truncated:
        resets += 1
        obs, info = env.reset()
print(f"300 steps  : reward {total:.2f} | resets {resets} | last score {info.get('score')}")
env.close()
print("HEADLESS OK")
"""
probe_path = pathlib.Path(WORK_DIR) / "probe_headless.py"
probe_path.write_text(PROBE)
run_logged(f'{PY} -u "{probe_path}" "{REPO_DIR}"', f"{WORK_DIR}/probe.log")

# The probe above is generated by this cell; if it ever needs changing, edit it here.

## E. Benchmark the number of parallel environments

The pygame simulation runs on the CPU, so the useful environment count depends entirely on
the machine. This cell measures *both* backends over several environment counts **and**
runs real PPO updates to measure end-to-end samples/s (stepping alone is meaningless when
the PPO update dominates the wall clock). `--frame-skip 4` makes the measurement match the
actual training shape.

The result is written to `BENCH_JSON` (outside the repository, so a later `git reset` in
section A cannot overwrite it), then the next cell picks the number to train with.

In [ ]:
# E.1 Measure.  Keep the candidate list bounded so this stays a few minutes.
candidates = sorted({1, 2, 4, 8, 16, 32} | ({64} if CPU_CORES >= 16 else set()))
bench_cmd = (
    f"{PY} -u training/benchmark_envs.py "
    f"--envs {','.join(str(c) for c in candidates)} "
    f"--steps 12000 --max-seconds 15 "
    f"--backends dummy,process "
    f"--frame-skip {FRAME_SKIP} --max-episode-steps {MAX_EPISODE_STEPS} "
    f"--n-steps {N_STEPS} --batch-size {BATCH_SIZE} "
    f"--ppo-probe 3 --ppo-steps 8192 --device auto "
    f'--json "{BENCH_JSON}"'
)
print("candidates:", candidates, "| vCPUs:", CPU_CORES)
run_logged(bench_cmd, f"{DRIVE_LOGS_DIR}/benchmark_{time.strftime('%Y%m%d_%H%M%S')}.log")

# the report itself also lands in the persistent history (it is a few KB)
drive_bench = pathlib.Path(DRIVE_LOGS_DIR) / "benchmark_envs.json"
shutil.copy2(BENCH_JSON, drive_bench)
print("benchmark report ->", drive_bench)


In [ ]:
# E.2 Read the report and choose N_ENVS.  Override FORCE_N_ENVS if you disagree.
FORCE_N_ENVS = None          # e.g. 16 to ignore the benchmark

report = json.loads(pathlib.Path(BENCH_JSON).read_text())
recommendation = report["recommendation"]

print(f"{'envs':>5} {'backend':>8} {'steps/s':>12} {'steps/s/env':>12} {'CPU %':>8} {'RAM MB':>9}")
for row in report["results"]:
    resources = row.get("resources") or {}
    print(f"{row['n_envs']:>5} {row['backend']:>8} {row['steps_per_sec']:>12,.0f} "
          f"{row['steps_per_sec_per_env']:>12,.1f} "
          f"{resources.get('cpu_percent', {}).get('mean', 0):>8.0f} "
          f"{resources.get('rss_mb', {}).get('mean', 0):>9.0f}")
print()
for probe in report.get("ppo_probes", []):
    if probe.get("error"):
        print(f"PPO probe {probe['n_envs']:>4} envs: ERROR {probe['error']}")
    else:
        print(f"PPO probe {probe['n_envs']:>4} envs: {probe['samples_per_sec']:>9,.0f} samples/s "
              f"({probe.get('env_share', 0) * 100:.0f}% stepping, "
              f"{100 * (1 - probe.get('env_share', 0)):.0f}% policy update)")

N_ENVS = int(FORCE_N_ENVS or recommendation["n_envs"])
BACKEND = str(recommendation["backend"])
BENCH_RATE = float(recommendation.get("steps_per_sec") or 0.0)

print()
print("recommendation:", json.dumps(recommendation, indent=2))
print()
print(f"-> training with N_ENVS={N_ENVS}, backend={BACKEND}"
      + ("  (forced)" if FORCE_N_ENVS else "  (measured)"))

try:
    import psutil
    estimate_mb = N_ENVS * 120
    available_mb = psutil.virtual_memory().available / 2 ** 20
    if estimate_mb > 0.5 * available_mb:
        print(f"WARNING: ~{N_ENVS} worker environments may need ~{estimate_mb:,.0f} MB of RAM "
              f"but only {available_mb:,.0f} MB is available -- consider FORCE_N_ENVS.")
except ImportError:
    pass

## F. Smoke test — the gate before any real training

Checks headless mode, reset/step, observation validity, the action meanings (jump, crouch,
fast-fall), reward anti-exploit properties, `check_env`, many independent environments,
process-vs-single-process agreement, a real PPO update, policy change, checkpoint
round-trip, CUDA placement when available and the visual code path.

**Do not continue if this does not end with `SMOKE TEST PASSED`.**

In [ ]:
# F. Smoke test (output is logged, the tail is printed, a failure raises).
text = run_logged(
    f"{PY} -u training/smoke_test.py --timesteps 4096 --envs 4 "
    f'--device auto --output "{WORK_DIR}/smoke"',
    f"{WORK_DIR}/smoke_test.log",
)
checks = [line for line in text.splitlines() if "checks passed" in line]
if "SMOKE TEST PASSED" not in text:
    raise RuntimeError("the smoke test did not pass -- do not start training. See the log above.")
print("GATE OPENED:", checks[-1] if checks else "all checks passed")

shutil.copy2(f"{WORK_DIR}/smoke_test.log", pathlib.Path(DRIVE_LOGS_DIR) / "smoke_test.log")
print("smoke test log   ->", f"{DRIVE_LOGS_DIR}/smoke_test.log")


## G. Short training run (200k–500k steps)

Proves that Colab can train *this* environment: the mean episode reward printed by the
evaluation callback should climb, and a model appears in `models/.../best/`.

The run lands in `SHORT_OUTPUT` on Google Drive, so even this wiring-check run keeps its
history; delete that folder in Drive if you do not want it.

Reference points from the local machine (3M steps, 32 envs, seed 42):
mean score **≈5,200** vs **≈189** for random actions and **≈12,300** for the scripted
heuristic. 300k steps is far too few to reach that — it is a wiring check, not a result.


In [ ]:
# G. Short PPO run: wiring check.  Safe to re-run (FRESH_SHORT_RUN clears the old output).
#    Lives on Drive (SHORT_OUTPUT), so even this throw-away run keeps its history.
FRESH_SHORT_RUN = True

if FRESH_SHORT_RUN and pathlib.Path(SHORT_OUTPUT).exists():
    shutil.rmtree(SHORT_OUTPUT)
    print("cleared", SHORT_OUTPUT)

eta = f" | estimate: {SHORT_TIMESTEPS / BENCH_RATE / 60:.0f} min" if BENCH_RATE else ""
print(f"environments: {N_ENVS} ({BACKEND}) | timesteps: {SHORT_TIMESTEPS:,}{eta}")
print("output      :", SHORT_OUTPUT)

run_logged(
    f'''{PY} -u training/train_ppo.py --timesteps {SHORT_TIMESTEPS} '''.rstrip()
    + f' --envs {N_ENVS} --backend {BACKEND} --seed {SEED} --device auto --progress '
    + f'{ppo_flags()} --eval-every 25000 --eval-episodes {EVAL_EPISODES} '
    + f'--checkpoint-every 100000 --output "{SHORT_OUTPUT}"',
    f"{DRIVE_LOGS_DIR}/short_run_{time.strftime('%Y%m%d_%H%M%S')}.log",
)


## H. Evaluate the short-run model

The existing evaluation system, on fixed seeds, deterministic actions, and the same
`--frame-skip 4` as training so the numbers are comparable with the local runs. It reports
the PPO agent **and** the `random`, `do_nothing` and `geometric_heuristic` baselines.

In [ ]:
# H. Evaluate the short model against the existing baselines.
run(
    f'{PY} -u training/evaluate_ppo.py --model "{SHORT_OUTPUT}" '
    f'--episodes {FINAL_EPISODES} --seed 1234 --frame-skip {FRAME_SKIP} '
    f'--json "{SHORT_OUTPUT}/evaluation.json"'
)

from training.artifacts import resolve_model_path

print("checkpoint used :", resolve_model_path(SHORT_OUTPUT))
short_eval = summarize_evaluation(f"{SHORT_OUTPUT}/evaluation.json", "SHORT RUN -- Colab wiring check")

## I. Inspect the persistent run (Drive)

Everything the long run produces lives in `DRIVE_OUTPUT_DIR` on Google Drive. This cell
shows the state a fresh runtime cares about: whether Drive is mounted, which checkpoint a
resumed run would continue from, how many steps are already trained, and where the next
checkpoints, logs and manifest will be written. Re-running it is always safe.


In [ ]:
# I. Show the persistent-run status (and mount Drive first if section B2 was skipped).
if not (pathlib.Path("/content/drive") / "MyDrive").is_dir():
    from google.colab import drive
    drive.mount("/content/drive")

pathlib.Path(LONG_OUTPUT).mkdir(parents=True, exist_ok=True)
pathlib.Path(DRIVE_LOGS_DIR).mkdir(parents=True, exist_ok=True)
state = locate_resume(LONG_OUTPUT)

DRIVE_STATUS = persistent_status()
try:
    DRIVE_STATUS["free_gib"] = round(shutil.disk_usage("/content/drive/MyDrive").free / 2 ** 30, 1)
except OSError:
    DRIVE_STATUS["free_gib"] = None

print("=" * 78)
print("PERSISTENT RUN STATUS")
print("=" * 78)
print(f"drive mounted         : {DRIVE_STATUS['drive_mounted']} (/content/drive)")
print(f"output directory      : {DRIVE_OUTPUT_DIR}")
print(f"logs directory        : {DRIVE_LOGS_DIR}")
print(f"existing checkpoint   : {'yes' if DRIVE_STATUS['checkpoint_found'] else 'no -- the next long run starts fresh'}")
print(f"resume source         : {state['source']}")
print(f"steps already trained : {state['trained']:,}")
print(f"periodic checkpoints  : {state['periodic']} | latest.zip: {state['latest'].is_file()} | best_model.zip: {state['best'].is_file()}")
print(f"next periodic saves   : {DRIVE_OUTPUT_DIR}/checkpoints/ (every 250k steps)")
print(f"next latest.zip       : {DRIVE_OUTPUT_DIR}/latest.zip (end of every session, incl. Ctrl-C)")
print(f"next best model       : {DRIVE_OUTPUT_DIR}/best/best_model.zip (whenever an evaluation improves)")
print(f"session manifest      : {DRIVE_OUTPUT_DIR}/session_manifest.jsonl")
try:
    print(f"drive free space      : {DRIVE_STATUS['free_gib']:.1f} GiB")
except TypeError:
    print("drive free space      : unknown")
previous = state["config"].get("ppo") or {}
if previous:
    print("previous config       :", {key: previous.get(key) for key in
          ("learning_rate", "n_steps", "batch_size", "n_epochs", "gamma", "ent_coef")})
    print("previous envs         :", state["config"].get("n_envs"), "| frame_skip:",
          state["config"].get("frame_skip"), "| seed:", state["config"].get("seed"))


## J. Long training (manual — nothing here starts by itself)

**This is the expensive cell.** Set `TOTAL_TIMESTEPS` to the total budget for the whole
run, then run the cell whenever you want training to happen.

* **Drive preflight:** the cell refuses to start unless Drive is mounted *and* writable,
  so it can never silently train into ephemeral `/content` storage.
* `--timesteps` means *additional* steps when resuming, so the cell subtracts what is
  already trained (read from `run_config.json` / the checkpoint names) and passes the
  remainder. Re-running it after a disconnect therefore continues instead of restarting.
* The full console output of the session goes to a timestamped log in `DRIVE_LOGS_DIR`,
  and every session appends one JSON line to `session_manifest.jsonl` — the complete
  experiment history stays inspectable on Drive.
* `USE_TENSORBOARD = True` (section 0) also writes SB3 event files to
  `{DRIVE_OUTPUT_DIR}/tensorboard` (a few MB per session at most).
* Checkpoints every 250k steps and the best evaluated model land in the Drive folder, so
  a disconnect costs at most one checkpoint interval. Periodic checkpoints are kept —
  nothing is pruned.
* `RESUME_LEARNING_RATE = None` keeps the learning rate stored in the checkpoint. Set it to
  `5e-4` (or lower, e.g. `1e-4`) only if you deliberately want to override it on resume.
* Set `FRESH_RUN = True` if you really want to throw the existing checkpoints away and
  start from zero (the old files are left on Drive as a backup).
* SB3 always finishes the rollout it is inside, so the run can overshoot
  `TOTAL_TIMESTEPS` by up to `n_steps × n_envs` steps. That is expected, not a bug.
* Progress streams into the session log, not the cell output — peek with
  `!tail -n 5 <log path>` in a scratch cell while it runs.

If the Colab runtime restarted: run 0 → A → B → C → E first (the benchmark must be
re-measured), then B2 → I → this cell.


In [ ]:
# J. LONG TRAINING -- manual.  Edit TOTAL_TIMESTEPS, then run this cell.
TOTAL_TIMESTEPS      = 10_000_000   # total budget for the whole run (not per session)
RESUME_LEARNING_RATE = None         # None = keep the checkpoint's learning rate on resume
FRESH_RUN            = False        # True = ignore existing checkpoints and start over

if "N_ENVS" not in globals():
    raise RuntimeError("run section E first (it benchmarks the environment count), "
                       "or set N_ENVS and BACKEND by hand, e.g. N_ENVS, BACKEND = 8, 'process'")

# ----------------------------------------------------------------- Drive preflight
# Fail here, clearly, instead of training for hours into ephemeral /content storage.
if not (pathlib.Path("/content/drive") / "MyDrive").is_dir():
    raise RuntimeError(
        "Google Drive is NOT mounted -- refusing to start the long run. "
        "Run section B2 (mount Drive) first, then this cell again."
    )
_write_probe = pathlib.Path(LONG_OUTPUT) / ".dino_write_probe"
_write_probe.write_text("ok")
_write_probe.unlink()

session_started = time.strftime("%Y-%m-%d %H:%M:%S")
state_before = locate_resume(LONG_OUTPUT)
if FRESH_RUN:
    resume_from, trained, source = None, 0, "fresh run (FRESH_RUN=True)"
else:
    resume_from, trained, source = state_before["resume_from"], state_before["trained"], state_before["source"]

remaining = int(TOTAL_TIMESTEPS) - int(trained)
tb_flag = f'--tensorboard-log "{DRIVE_OUTPUT_DIR}/tensorboard"' if USE_TENSORBOARD else ""
print(f"drive mounted    : yes (persistent output: {DRIVE_OUTPUT_DIR})")
print(f"resume           : {resume_from or 'nothing yet'}  [{source}]")
print(f"trained so far   : {trained:,} steps")
print(f"remaining budget : {max(0, remaining):,} steps of {TOTAL_TIMESTEPS:,}")
print(f"environments     : {N_ENVS} ({BACKEND}) | seed {SEED}")
print(f"tensorboard      : {DRIVE_OUTPUT_DIR}/tensorboard" if USE_TENSORBOARD
      else "tensorboard      : off (USE_TENSORBOARD=False)")
if BENCH_RATE:
    print(f"estimated time   : {max(0, remaining) / BENCH_RATE / 3600:.1f} h "
          f"(benchmark: {BENCH_RATE:,.0f} PPO samples/s -- an estimate, not a promise)")

if remaining <= 0:
    print("\nTOTAL_TIMESTEPS already reached -- nothing to train. Evaluate in section K,")
    print("or raise TOTAL_TIMESTEPS if you want to keep going.")
else:
    # A fresh run starts from LEARNING_RATE; a resume keeps the checkpoint's value unless
    # RESUME_LEARNING_RATE says otherwise.
    flags = ppo_flags(RESUME_LEARNING_RATE if resume_from else LEARNING_RATE)
    resume_arg = f'--resume "{resume_from}"' if resume_from else ""
    session_log = f"{DRIVE_LOGS_DIR}/train_{time.strftime('%Y%m%d_%H%M%S')}.log"
    print(f"\ncheckpoints -> {LONG_OUTPUT}/checkpoints (Google Drive)")
    print(f"console log -> {session_log}")
    print("Re-run this cell after a disconnect to continue; Ctrl-C stops after saving latest.zip.")
    print("Progress is written to the session log -- peek with !tail -n 5 <log path>.\n")
    run_logged(
        f'''{PY} -u training/train_ppo.py --timesteps {remaining} {resume_arg} '''.rstrip()
        + f' --envs {N_ENVS} --backend {BACKEND} --seed {SEED} --device auto --progress {flags} {tb_flag} '
        + f'--eval-every 250000 --eval-episodes {EVAL_EPISODES} --checkpoint-every 250000 '
        + f'--output "{LONG_OUTPUT}"',
        session_log,
    )
    state_after = locate_resume(LONG_OUTPUT)
    write_session_manifest({
        "session_start": session_started,
        "session_end": time.strftime("%Y-%m-%d %H:%M:%S"),
        "resume_from": str(resume_from) if resume_from else None,
        "steps_before": int(state_before["trained"]),
        "steps_after": int(state_after["trained"]),
        "requested_total": int(TOTAL_TIMESTEPS),
        "n_envs": N_ENVS,
        "backend": BACKEND,
        "seed": SEED,
        "tensorboard": bool(USE_TENSORBOARD),
        "log": session_log,
    })
    DRIVE_STATUS.update({
        "checkpoint_found": True,
        "resume_source": state_after["source"],
        "steps_trained": int(state_after["trained"]),
    })
    print(f"steps after session : {state_after['trained']:,}")
    print(f"latest.zip          : {state_after['latest']}")
    print(f"best model          : {state_after['best']} "
          f"({'present' if state_after['best'].is_file() else 'none yet'})")


## K. Evaluate the final model

Same evaluation code and seeds as section H, so the short run and the long run are
directly comparable. `best/best_model.zip` is the best *evaluated* checkpoint;
`latest.zip` is the most recent state. Both are reported when they differ.

In [ ]:
# K. Evaluate the final model (best evaluated checkpoint + baselines).
run(
    f'{PY} -u training/evaluate_ppo.py --model "{LONG_OUTPUT}" '
    f'--episodes {FINAL_EPISODES} --seed 1234 --frame-skip {FRAME_SKIP} '
    f'--json "{LONG_OUTPUT}/evaluation.json"'
)

from training.artifacts import resolve_model_path

best_checkpoint = resolve_model_path(LONG_OUTPUT, prefer="best")
latest_checkpoint = resolve_model_path(LONG_OUTPUT, prefer="latest")
final_eval = summarize_evaluation(f"{LONG_OUTPUT}/evaluation.json", "FINAL MODEL (best checkpoint)")
print("best checkpoint  :", best_checkpoint)
print("latest checkpoint:", latest_checkpoint)

if latest_checkpoint != best_checkpoint:
    run(
        f'{PY} -u training/evaluate_ppo.py --model "{latest_checkpoint}" '
        f'--episodes {FINAL_EPISODES} --seed 1234 --frame-skip {FRAME_SKIP} --no-baselines '
        f'--json "{LONG_OUTPUT}/evaluation_latest.json"'
    )
    summarize_evaluation(f"{LONG_OUTPUT}/evaluation_latest.json", "LATEST CHECKPOINT (most recent state)")

## L. Package and download

The run already lives on Google Drive; this copies the essentials into `/content` and zips
them so you can download the model without opening Drive.

In [ ]:
# L. Package the model + config + evaluation, then download the zip.
import zipfile

INCLUDE_CHECKPOINTS = False      # True also packs every periodic checkpoint (~215 KB each)

stage = pathlib.Path(WORK_DIR) / "dino_run_ppo_results"
if stage.exists():
    shutil.rmtree(stage)
stage.mkdir(parents=True, exist_ok=False)

output = pathlib.Path(LONG_OUTPUT)
copied = []
for relative in ["run_config.json", "latest.zip", "best/best_model.zip", "evaluations.npz",
                 "evaluation.json", "evaluation_latest.json"]:
    source = output / relative
    if source.is_file():
        destination = stage / relative
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source, destination)
        copied.append(relative)
if INCLUDE_CHECKPOINTS and (output / "checkpoints").is_dir():
    shutil.copytree(output / "checkpoints", stage / "checkpoints")
    copied.append("checkpoints/ (all periodic snapshots)")

zip_path = pathlib.Path(WORK_DIR) / "dino_run_ppo_results.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(stage.rglob("*")):
        if path.is_file():
            archive.write(path, path.relative_to(stage))

print("packaged:", copied)
print(f"zip      : {zip_path} ({zip_path.stat().st_size / 1e6:.1f} MB)")

from google.colab import files
files.download(str(zip_path))

## M. Run the trained model in the real game (on your machine)

The model is a normal Stable-Baselines3 PPO checkpoint. It is **resolution independent**:
trained at 800×600, it also plays a 1024×768 or a 2560×1440 window. Nothing about the game
itself changed — `python main.py` is still the same game you play by hand.

In [ ]:
# M. The exact commands for your own machine (copy-paste, no Colab involved).
run_name = pathlib.Path(LONG_OUTPUT).name
print(f"""
=============================================================================
 Bring the model back to your machine
=============================================================================
 1. Download "dino_run_ppo_results.zip" (section L) -- or copy the folder
    straight out of Google Drive: MyDrive/dino_run_ppo/{run_name}/

 2. In your checkout of {REPO_URL.split('/')[-1].replace('.git','')}, unpack it:

      unzip dino_run_ppo_results.zip -d models/{run_name}

 3. WATCH IT PLAY the real game (real window, real physics):

      python training/play_agent.py --model models/{run_name}

 4. Reproduce these Colab numbers locally:

      python training/evaluate_ppo.py --model models/{run_name} --episodes 20 --seed 1234 --frame-skip 4

 5. Off-screen sanity check on any machine (no window needed):

      python training/play_agent.py --model models/{run_name} --selftest --frames 1200

 6. Play the game yourself, unchanged:

      python main.py

 Files that matter inside the folder:
      best/best_model.zip   <- best evaluated policy (what evaluation/playback load)
      latest.zip            <- most recent state (what --resume continues from)
      run_config.json       <- hyperparameters, obs/action/reward description, seeds
      evaluation.json       <- the numbers printed above
=============================================================================
""")

## Notes and troubleshooting

**What this notebook does *not* change.** The game, the physics, the observation
(`dino-obs-v1`, a 47-float Box), the reward and the action space
(0 = nothing, 1 = jump, 2 = down — crouch on the ground, fast-fall in the air) are exactly
the code in the repository. Only the PPO hyperparameters are passed explicitly, and they are
the values this project was verified with.

**CPU vs GPU.** The simulation is pygame, so it runs on the CPU — in worker processes, one
worker per core (`--backend process`). The GPU only runs a ~15k-parameter MLP. A GPU is
nice, but many environments on a fast CPU matter more; that is why section E measures
throughput instead of copying a number from another machine.

**Resuming.** `--timesteps` is *additional* steps when `--resume` is used, which is why
section J computes `remaining = TOTAL_TIMESTEPS - trained`. `best/best_model.zip` never
regresses: `training/train_ppo.py` records the best evaluation in `run_config.json` and
restores it on resume. Every session also appends one JSON line to `session_manifest.jsonl`
and a timestamped console log to `MyDrive/dino_run_ppo/logs/`, so the whole experiment
history stays inspectable from Drive.

**Where things are written.**

| what | where | survives a disconnect? |
|---|---|---|
| code | `/content/Dino-Run` | no (re-cloned in section A) |
| probe / smoke / benchmark scratch | `/content/dino_run_work` | no (report + smoke log copied to `DRIVE_LOGS_DIR`) |
| the short wiring-check run | `MyDrive/dino_run_ppo/ppo_dino_colab_short` | **yes** |
| the long run (checkpoints, best model, config, evaluations, tensorboard) | `MyDrive/dino_run_ppo/ppo_dino_colab` | **yes** |
| session console logs + manifest | `MyDrive/dino_run_ppo/logs/` + `.../session_manifest.jsonl` | **yes** |

**Common problems**

* *"the clone is missing ..."* — the RL code is not on `origin/main` yet. Push it locally,
  then re-run section A.
* *Smoke test fails on "display mode is 1x1"* — something opened a pygame window earlier in
  the session. Restart the runtime and run the notebook from the top.
* *`ModuleNotFoundError: game`* — you ran a command from the wrong directory. Every cell in
  this notebook runs with `cwd` set to the repository root for you.
* *"Google Drive is NOT mounted"* — section J refuses to train into ephemeral storage.
  Run section B2 (or I), then section J again.
* *The benchmark recommends a small number* (2–8 on a free T4 runtime, which has ~2 vCPUs) —
  that is a measurement, not a bug: with the rollout update included, more environments stop
  helping. Accept it, or set `FORCE_N_ENVS` and compare.
* *Training stops when the browser tab closes / after hours of inactivity* — that is Colab.
  Re-run sections 0 → A → B → C → E → I → J; training continues from the last Drive
  checkpoint.
* *Long runs on the free tier* — 10M steps may take many hours and need several sessions.
  The resume logic is designed for exactly that: just run section J again.

**Reference numbers from the local machine** (3M steps, 32 envs, seed 42, 800×600,
`--frame-skip 4`, 20 evaluation episodes on seeds 1234+):

| policy | mean score | median | best | survival |
|---|---|---|---|---|
| PPO (best checkpoint) | 5,192 | 520 | 24,000 | 86.5 s |
| random actions | 189 | 182 | 267 | 3.1 s |
| do nothing | 181 | 181 | 181 | 3.0 s |
| scripted `geometric_heuristic` | 12,331 | 12,454 | 24,000 | 205.5 s |

The PPO policy is bimodal: ~16/20 episodes died early on a mistimed low-speed hop, 4/20
survived the full 400 s cap. Beating the scripted heuristic is the real bar.
